# Groundwater Well Placement — Simulated Annealing
## Thiem equation: $s = \dfrac{Q\,\ln(r/R)}{2\pi T}$

**Variables**

| Symbol | Meaning | Unit |
|--------|---------|------|
| $s$ | head change at observation point ($s<0$ = drawdown, $s>0$ = mounding) | m |
| $Q$ | pumping rate ($Q>0$ = extraction, $Q<0$ = injection) | m³/day |
| $r$ | radial distance from well (polar coordinate, $\theta = 0$) | m |
| $R$ | radius of influence | m |
| $T$ | aquifer transmissivity | m²/day |

**Scenario** — hydraulic containment of a rectangular contaminant plume.
We jointly optimise **well positions** (row, col on a grid) and **pumping rates** $Q$ using Simulated Annealing.

---


In [ ]:
# !pip install timml -q

In [ ]:
import timml
import copy, math, random, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
warnings.filterwarnings("ignore")
%matplotlib inline
plt.rcParams.update({"figure.dpi": 130, "font.size": 10})
print("Libraries loaded ✓")


## ⚙️ Configuration — edit this cell to change the scenario


In [ ]:
# ── Grid ──────────────────────────────────────────────────────────────────────
GRID_NX   = 30        # columns
GRID_NY   = 30        # rows
CELL_SIZE = 10.0      # [m] physical size of each cell

# ── Aquifer ────────────────────────────────────────────────────────────────────
TRANSMISSIVITY = 100.0   # [m²/day]  T
R_INFLUENCE    = 1000.0   # [m]       radius of influence R
WELLBORE_R     = 0.05    # [m]       minimum r (avoids ln singularity)

# ── Well fleet ─────────────────────────────────────────────────────────────────
N_EXTRACT = 4            # number of extraction wells
N_INJECT  = 2            # number of injection wells

Q_E_MIN, Q_E_MAX, Q_E_INIT = 100.0,  2000.0,  800.0   # [m³/day] extraction
Q_I_MIN, Q_I_MAX, Q_I_INIT = -2000.0, -100.0, -400.0   # [m³/day] injection
Q_STEP    = 50.0        # discrete step for Q perturbation

# ── Plume zone (grid indices, inclusive) ───────────────────────────────────────
PLUME_ROWS = (10, 20)
PLUME_COLS = (10, 20)

# ── Physical targets ───────────────────────────────────────────────────────────
MIN_DRAW_TARGET       = 1.0   # [m] minimum required drawdown inside plume
BOUNDARY_MAX_DRAWDOWN = 1.5   # [m] max tolerated drawdown at boundary nodes

# ── Objective weights ──────────────────────────────────────────────────────────
W_CONTAINMENT = 1.0   # reward deep drawdown in plume
W_BOUNDARY    = 4.0   # penalise over-drawing boundary
W_ENERGY      = 20.0  # penalise unnecessary total pumping
W_PROXIMITY   = 1.5   # keep wells spread out
PROXIMITY_MIN = 4.0   # [cells] desired minimum inter-well spacing

# ── SA schedule ────────────────────────────────────────────────────────────────
SA_T0       = 2.0
SA_T_MIN    = 1e-4
SA_ALPHA    = 0.997
SA_MAX_ITER = 12_000
SA_SEED     = 42

# ── Drawdown field method ───────────────────────────────────────────────────────
USE_TIMML = False     # True for TimML, if False then the Thiem Equation is used


print(f"Grid: {GRID_NX}×{GRID_NY}  ({GRID_NX*CELL_SIZE:.0f}×{GRID_NY*CELL_SIZE:.0f} m)")
print(f"T = {TRANSMISSIVITY} m²/day,  R = {R_INFLUENCE} m")
print(f"Extraction wells: {N_EXTRACT}  |  Injection wells: {N_INJECT}")


## 🗺️ Spatial Masks
Pre-compute which grid nodes belong to the plume zone and the boundary ring.


In [ ]:
def make_plume_mask(ny, nx, row_range, col_range):
    m = np.zeros((ny, nx), dtype=bool)
    m[row_range[0]:row_range[1]+1, col_range[0]:col_range[1]+1] = True
    return m

def make_boundary_mask(ny, nx):
    m = np.zeros((ny, nx), dtype=bool)
    m[0, :] = m[-1, :] = m[:, 0] = m[:, -1] = True
    return m

PLUME_MASK    = make_plume_mask(GRID_NY, GRID_NX, PLUME_ROWS, PLUME_COLS)
BOUNDARY_MASK = make_boundary_mask(GRID_NY, GRID_NX)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
for ax, mask, title in zip(axes,
                            [PLUME_MASK, BOUNDARY_MASK],
                            ["Plume zone", "Boundary ring"]):
    ax.imshow(mask, origin="lower", cmap="YlOrRd", aspect="equal")
    ax.set_title(title)
    ax.set_xlabel("col"); ax.set_ylabel("row")
plt.tight_layout()
plt.show()
print(f"Plume nodes: {PLUME_MASK.sum()}  |  Boundary nodes: {BOUNDARY_MASK.sum()}")


## 💧 Thiem Equation & Superposition

$$s(r) = \frac{Q \ln(r/R)}{2\pi T}$$

For **multiple wells** (superposition principle):

$$s_{\text{total}}(x,y) = \sum_i \frac{Q_i\,\ln(r_i/R)}{2\pi T}$$

where $r_i = \sqrt{(x-x_i)^2 + (y-y_i)^2}$ is the polar radius to well $i$.


In [ ]:
def drawdown_field(wells, ny=None, nx=None, cell=None, T=None, R=None, use_timml=None):
    use_timml = use_timml or USE_TIMML
    
    if use_timml:
        ny  = ny   or GRID_NY
        nx  = nx   or GRID_NX
        cell= cell or CELL_SIZE
        T   = T    or TRANSMISSIVITY
        # R   = R    or R_INFLUENCE
        xr  = 1E4
    
        # X, Y = np.meshgrid(np.arange(nx)*cell, np.arange(ny)*cell)
        s    = np.zeros((ny, nx))
    
        model = timml.ModelMaq(kaq=T, z=[1, 0])
        timml_wells = [timml.Well(model=model,
                        xw=data['col'] * cell,
                        yw=data['row'] * cell,
                        Qw=data['Q'],
                        rw=WELLBORE_R) for data in wells]
    
        constant = timml.Constant(model=model, xr=xr, yr=xr, hr=0)
        model.solve(silent=True)
        return model.headgrid(np.arange(nx)*cell, np.arange(ny)*cell, 0).squeeze()
    else:
        """
        Vectorised Thiem superposition.
        Returns s[ny, nx] — head change [m].  s<0 = drawdown, s>0 = mounding.
        """
        ny  = ny   or GRID_NY
        nx  = nx   or GRID_NX
        cell= cell or CELL_SIZE
        T   = T    or TRANSMISSIVITY
        R   = R    or R_INFLUENCE
    
        X, Y = np.meshgrid(np.arange(nx)*cell, np.arange(ny)*cell)
        s    = np.zeros((ny, nx))
    
        for w in wells:
            wx, wy = w["col"]*cell, w["row"]*cell
            r      = np.maximum(np.sqrt((X-wx)**2 + (Y-wy)**2), WELLBORE_R)
            mask   = r < R
            s[mask] += w["Q"] * np.log(r[mask]/R) / (2*math.pi*T)
        return s
        

In [ ]:
# ── Quick single-well demo ─────────────────────────────────────────────────────
demo_well = [{"row": 15, "col": 15, "Q": 1000, "type": "extraction"}]
s_demo    = drawdown_field(demo_well, use_timml=False)
dm_demo   = -s_demo   # drawdown magnitude

r_vals  = np.linspace(WELLBORE_R, R_INFLUENCE, 300)
s_curve = [demo_well[0]["Q"] * math.log(r/R_INFLUENCE) / (2*math.pi*TRANSMISSIVITY)
           for r in r_vals]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

im = ax1.imshow(dm_demo, origin="lower", cmap="Blues",
                extent=[0, GRID_NX*CELL_SIZE, 0, GRID_NY*CELL_SIZE])
plt.colorbar(im, ax=ax1, label="Drawdown magnitude –s [m]")
ax1.plot(15*CELL_SIZE+CELL_SIZE/2, 15*CELL_SIZE+CELL_SIZE/2,
         "rv", ms=12, label="Well (Q=1000 m³/d)")
ax1.set_title("Thiem drawdown cone (single extraction well)")
ax1.set_xlabel("x [m]"); ax1.set_ylabel("y [m]"); ax1.legend()

ax2.plot(r_vals, [-s for s in s_curve], color="steelblue", lw=2)
ax2.axvline(R_INFLUENCE, color="red", ls="--", label=f"R = {R_INFLUENCE} m")
ax2.set_xlabel("r  [m]"); ax2.set_ylabel("Drawdown –s  [m]")
ax2.set_title("Thiem curve  Q=1000, T=100")
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout(); plt.show()

In [ ]:
# ── Quick single-well demo ─────────────────────────────────────────────────────
demo_well = [{"row": 15, "col": 15, "Q": 1000, "type": "extraction"}]
s_demo    = drawdown_field(demo_well, use_timml=True)
dm_demo   = -s_demo   # drawdown magnitude

r_vals  = np.linspace(WELLBORE_R, R_INFLUENCE, 300)
s_curve = [demo_well[0]["Q"] * math.log(r/R_INFLUENCE) / (2*math.pi*TRANSMISSIVITY)
           for r in r_vals]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

im = ax1.imshow(dm_demo, origin="lower", cmap="Blues",
                extent=[0, GRID_NX*CELL_SIZE, 0, GRID_NY*CELL_SIZE])
plt.colorbar(im, ax=ax1, label="Drawdown magnitude –s [m]")
ax1.plot(15*CELL_SIZE+CELL_SIZE/2, 15*CELL_SIZE+CELL_SIZE/2,
         "rv", ms=12, label="Well (Q=1000 m³/d)")
ax1.set_title("Thiem drawdown cone (single extraction well)")
ax1.set_xlabel("x [m]"); ax1.set_ylabel("y [m]"); ax1.legend()

ax2.plot(r_vals, [-s for s in s_curve], color="steelblue", lw=2)
ax2.axvline(R_INFLUENCE, color="red", ls="--", label=f"R = {R_INFLUENCE} m")
ax2.set_xlabel("r  [m]"); ax2.set_ylabel("Drawdown –s  [m]")
ax2.set_title("Thiem curve  Q=1000, T=100")
ax2.legend(); ax2.grid(alpha=0.3)

# plt.tight_layout(); plt.show()

## 🎯 Objective Function

$$\text{cost} = \underbrace{-W_c \cdot \text{containment}}_\text{reward}
              + \underbrace{W_b \cdot \text{boundary penalty}}_\text{constraint}
              + \underbrace{W_e \cdot \text{energy}}_\text{efficiency}
              + \underbrace{W_p \cdot \text{proximity}}_\text{spacing}$$

| Term | Formula | Purpose |
|------|---------|---------|
| containment | $\text{mean}\bigl(\max(0,\,-s-s^*)\bigr)$ over plume | reward sufficient drawdown in plume |
| boundary | $\text{mean}\bigl(\max(0,\,-s-s_b)\bigr)$ over boundary | protect outer supply ring |
| energy | $\sum|Q_i|\,/\,Q_{\max,\text{total}}$ | discourage over-pumping |
| proximity | $\sum_{i<j}\max(0,\,d^*-d_{ij})\,/\,d^*$ | keep wells spread apart |


In [ ]:
_Q_TOTAL_MAX = float(N_EXTRACT*Q_E_MAX) # + N_INJECT*abs(Q_I_MIN))

def compute_cost(wells):
    s  = drawdown_field(wells)
    dm = -s   # drawdown magnitude (positive = drawdown)

    containment = float(np.mean(np.maximum(0.0, dm[PLUME_MASK]    - MIN_DRAW_TARGET)))
    bnd_excess  = float(np.mean(np.maximum(0.0, dm[BOUNDARY_MASK] - BOUNDARY_MAX_DRAWDOWN)))
    energy      = sum(w["Q"] for w in wells if w["Q"] >= 0) / _Q_TOTAL_MAX
    pump_total  = sum(w["Q"] for w in wells if w["Q"] >= 0)
    injection_total   = sum(w["Q"] for w in wells if w["Q"] < 0)
    Q_diff = np.maximum(0.0, abs(pump_total + injection_total))

    prox = 0.0
    for i in range(len(wells)):
        for j in range(i+1, len(wells)):
            d = math.sqrt((wells[i]["row"]-wells[j]["row"])**2
                         +(wells[i]["col"]-wells[j]["col"])**2)
            if d < PROXIMITY_MIN:
                prox += (PROXIMITY_MIN - d) / PROXIMITY_MIN

    return (-W_CONTAINMENT * containment
            + W_BOUNDARY   * bnd_excess
            + W_ENERGY     * energy
            + W_PROXIMITY  * prox
            + 10000        * Q_diff)


# ── Sanity-check cost on a hand-crafted configuration ────────────────────────
test_wells = [
    {"row": 15, "col": 12, "Q": 1000, "type": "extraction"},
    {"row": 15, "col": 18, "Q": 1000, "type": "extraction"},
    {"row":  2, "col":  2, "Q": -200, "type": "injection"},
]
print(f"Test cost: {compute_cost(test_wells):.4f}   "
      f"(lower is better; purely illustrative)")


## 🔥 Simulated Annealing

Three neighbourhood moves, chosen at random each iteration:

| Move | Probability | Action |
|------|------------|--------|
| **A — Relocate** | 0.45 | Move one well to a random empty cell; Q unchanged |
| **B — Swap** | 0.10 | Exchange (row,col) of two wells; types & Q travel with each well |
| **C — Rate** | 0.45 | Perturb Q of one well by ±1–2 steps of `Q_STEP` |

Acceptance: Metropolis criterion $P(\text{accept}) = \exp(-\Delta\text{cost}/T)$.
Cooling: geometric $T_{k+1} = \alpha\,T_k$.


In [ ]:
def _random_initial():
    positions, wells = set(), []
    for _ in range(N_EXTRACT):
        while True:
            r, c = random.randint(0, GRID_NY-1), random.randint(0, GRID_NX-1)
            if (r,c) not in positions:
                positions.add((r,c))
                wells.append({"row":r,"col":c,"Q":Q_E_INIT,"type":"extraction"})
                break
    for _ in range(N_INJECT):
        while True:
            r, c = random.randint(0, GRID_NY-1), random.randint(0, GRID_NX-1)
            if (r,c) not in positions:
                positions.add((r,c))
                wells.append({"row":r,"col":c,"Q":Q_I_INIT,"type":"injection"})
                break
    return wells

def _perturb_Q(Q, wtype):
    delta = random.choice([-2,-1,1,2]) * Q_STEP
    if wtype == "extraction":
        return max(Q_E_MIN, min(Q_E_MAX, Q+delta))
    return max(Q_I_MIN, min(Q_I_MAX, Q+delta))

def _neighbour(wells):
    nw   = copy.deepcopy(wells)
    roll = random.random()
    if roll < 0.45:                            # Move A: relocate
        occupied = {(w["row"],w["col"]) for w in nw}
        idx = random.randrange(len(nw))
        occupied.discard((nw[idx]["row"], nw[idx]["col"]))
        for _ in range(100):
            r, c = random.randint(0,GRID_NY-1), random.randint(0,GRID_NX-1)
            if (r,c) not in occupied:
                nw[idx]["row"], nw[idx]["col"] = r, c; break
    elif roll < 0.55 and len(nw) >= 2:         # Move B: swap
        i, j = random.sample(range(len(nw)), 2)
        nw[i]["row"],nw[j]["row"] = nw[j]["row"],nw[i]["row"]
        nw[i]["col"],nw[j]["col"] = nw[j]["col"],nw[i]["col"]
    else:                                       # Move C: perturb Q
        idx = random.randrange(len(nw))
        nw[idx]["Q"] = _perturb_Q(nw[idx]["Q"], nw[idx]["type"])
    return nw

def run_sa(T0=SA_T0, T_min=SA_T_MIN, alpha=SA_ALPHA,
           max_iter=SA_MAX_ITER, seed=SA_SEED, verbose=True):
    """
    Run the SA optimiser.
    Returns best_wells, best_cost, cost_history, temp_history, q_history.
    """
    random.seed(seed); np.random.seed(seed)

    current = _random_initial()
    cur_cost = compute_cost(current)
    best, best_cost = copy.deepcopy(current), cur_cost

    T = T0
    cost_hist, temp_hist, q_hist = [cur_cost], [T], [sum(abs(w["Q"]) for w in current)]
    n_accept = 0

    for k in range(1, max_iter+1):
        cand      = _neighbour(current)
        cand_cost = compute_cost(cand)
        delta     = cand_cost - cur_cost

        if delta < 0 or random.random() < math.exp(-delta/T):
            current, cur_cost = cand, cand_cost
            n_accept += 1
            if cur_cost < best_cost:
                best, best_cost = copy.deepcopy(current), cur_cost

        T = max(T*alpha, T_min)
        cost_hist.append(cur_cost)
        temp_hist.append(T)
        q_hist.append(sum(abs(w["Q"]) for w in current))

        if verbose and k % (max_iter//6) == 0:
            print(f"  iter {k:>6d} | T={T:.5f} | best={best_cost:.4f} "
                  f"| accept={100*n_accept/k:.1f}%")

    if verbose:
        print(f"\nFinal best cost : {best_cost:.4f}")
        print(f"Accepted        : {n_accept}/{max_iter} ({100*n_accept/max_iter:.1f}%)")
    return best, best_cost, cost_hist, temp_hist, q_hist

print("SA functions defined ✓")


## ▶️ Run the Optimiser


In [ ]:
best_wells, best_cost, cost_hist, temp_hist, q_hist = run_sa()


## 📋 Results Summary


In [ ]:
s_opt = drawdown_field(best_wells)
dm    = -s_opt
plume_dm = dm[PLUME_MASK]
bnd_dm   = dm[BOUNDARY_MASK]
total_Q_pump  = sum(abs(w["Q"]) for w in best_wells if w["Q"] >= 0)
total_Q_inj  = sum(abs(w["Q"]) for w in best_wells if w["Q"] <= 0)
coverage = float(np.mean(plume_dm >= MIN_DRAW_TARGET)) * 100

print(f"{'Well':<4} {'Type':<12} {'row':>4} {'col':>4}  "
      f"{'x [m]':>6} {'y [m]':>6}  {'Q [m³/d]':>10}")
print("─"*55)
for i, w in enumerate(best_wells, 1):
    print(f"{i:<4} {w['type']:<12} {w['row']:>4} {w['col']:>4}  "
          f"{w['col']*CELL_SIZE:>6.0f} {w['row']*CELL_SIZE:>6.0f}  {w['Q']:>10.0f}")
print("─"*55)
print(f"\nPlume drawdown : min={plume_dm.min():.2f}  "
      f"mean={plume_dm.mean():.2f}  max={plume_dm.max():.2f} m")
print(f"Boundary draw  : min={bnd_dm.min():.2f}  "
      f"mean={bnd_dm.mean():.2f}  max={bnd_dm.max():.2f} m")
print(f"Coverage       : {coverage:.1f}% of plume nodes ≥ {MIN_DRAW_TARGET} m target")
print(f"Total Q_pump      : {total_Q_pump:.0f} m³/day")
print(f"Total Q_inj       : {total_Q_inj:.0f} m³/day")


## 🗺️ Drawdown Field & Convergence


In [ ]:
def plot_overview(wells, cost_h, temp_h, q_h):
    s  = drawdown_field(wells)
    dm = -s
    iters = np.arange(len(cost_h))

    fig = plt.figure(figsize=(18, 11))
    fig.suptitle(
        "SA — Optimal Well Placement & Pumping Rates\n"
        r"$s = Q\,\ln(r/R)\,/\,(2\pi T)$"
        f"     T={TRANSMISSIVITY} m²/day   R={R_INFLUENCE} m",
        fontsize=13, fontweight="bold")
    gs = GridSpec(3, 3, figure=fig, hspace=0.50, wspace=0.38)

    # ── Map ──────────────────────────────────────────────────────────────── #
    ax_m = fig.add_subplot(gs[:, :2])
    ext  = [0, GRID_NX*CELL_SIZE, 0, GRID_NY*CELL_SIZE]
    vlim = max(abs(float(s.min())), abs(float(s.max())), 0.1)
    im   = ax_m.imshow(s, origin="lower", extent=ext, cmap="RdBu_r",
                       vmin=-vlim, vmax=vlim, aspect="equal",
                       interpolation="bilinear")
    cb   = plt.colorbar(im, ax=ax_m, fraction=0.032, pad=0.02)
    cb.set_label("Head change s [m]\n(– = drawdown, + = mounding)", fontsize=9)

    # contours of drawdown magnitude
    X2, Y2 = np.meshgrid(np.arange(GRID_NX)*CELL_SIZE,
                          np.arange(GRID_NY)*CELL_SIZE)
    cs = ax_m.contour(X2, Y2, dm, levels=10, colors="black",
                      linewidths=0.5, alpha=0.5)
    ax_m.clabel(cs, inline=True, fontsize=7, fmt="%.1f m")

    # plume box
    ax_m.add_patch(mpatches.FancyBboxPatch(
        (PLUME_COLS[0]*CELL_SIZE, PLUME_ROWS[0]*CELL_SIZE),
        (PLUME_COLS[1]-PLUME_COLS[0])*CELL_SIZE,
        (PLUME_ROWS[1]-PLUME_ROWS[0])*CELL_SIZE,
        boxstyle="round,pad=0", lw=2.2, edgecolor="gold",
        facecolor="gold", alpha=0.18))

    for w in wells:
        xw = w["col"]*CELL_SIZE + CELL_SIZE/2
        yw = w["row"]*CELL_SIZE + CELL_SIZE/2
        mk, col = ("v","crimson") if w["type"]=="extraction" else ("^","deepskyblue")
        ax_m.plot(xw, yw, mk, ms=14, color=col, mec="white", mew=1.5, zorder=5)
        ax_m.annotate(f"{w['Q']:.0f}", (xw,yw), fontsize=7.5, ha="center",
                      va="bottom", color="white", fontweight="bold",
                      xytext=(0,12), textcoords="offset points")

    for sp in ax_m.spines.values():
        sp.set_edgecolor("darkorange"); sp.set_linewidth(2.5)
    ax_m.legend(handles=[
        mpatches.Patch(fc="gold", alpha=0.5, ec="gold", label="Plume"),
        plt.Line2D([0],[0],marker="v",color="w",mfc="crimson",ms=12,
                   label="Extraction"),
        plt.Line2D([0],[0],marker="^",color="w",mfc="deepskyblue",ms=12,
                   label="Injection"),
        mpatches.Patch(fc="none",ec="darkorange",lw=2.5,label="Boundary"),
    ], loc="upper left", fontsize=9, framealpha=0.85)
    ax_m.set_xlabel("x [m]"); ax_m.set_ylabel("y [m]")
    ax_m.set_title("Superimposed Thiem field (optimal)")
    ax_m.grid(color="white", alpha=0.15, lw=0.4)

    # ── Cost ─────────────────────────────────────────────────────────────── #
    ax_c = fig.add_subplot(gs[0,2])
    ax_c.plot(iters, cost_h, lw=0.8, color="steelblue", alpha=0.7)
    ax_c.plot(iters, np.minimum.accumulate(cost_h), lw=1.8,
              color="darkblue", ls="--", label="Running min")
    ax_c.set_xlabel("Iteration"); ax_c.set_ylabel("Cost")
    ax_c.set_title("Convergence"); ax_c.legend(fontsize=8); ax_c.grid(alpha=0.3)

    # ── Temperature ───────────────────────────────────────────────────────── #
    ax_t = fig.add_subplot(gs[1,2])
    ax_t.semilogy(iters, temp_h, lw=1.2, color="firebrick")
    ax_t.set_xlabel("Iteration"); ax_t.set_ylabel("Temperature (log)")
    ax_t.set_title("Cooling Schedule"); ax_t.grid(alpha=0.3, which="both")

    # ── ΣQ ───────────────────────────────────────────────────────────────── #
    ax_q = fig.add_subplot(gs[2,2])
    ax_q.plot(iters, q_h, lw=0.9, color="seagreen", alpha=0.8)
    ax_q.plot(iters, np.minimum.accumulate(q_h), lw=1.8,
              color="darkgreen", ls="--", label="Running min")
    ax_q.set_xlabel("Iteration"); ax_q.set_ylabel("Σ|Q| [m³/day]")
    ax_q.set_title("Total Pumping Rate"); ax_q.legend(fontsize=8); ax_q.grid(alpha=0.3)

    plt.tight_layout(); plt.show()

plot_overview(best_wells, cost_hist, temp_hist, q_hist)


## 📊 Centreline Profile
Drawdown magnitude along the plume centreline row.


In [ ]:
def plot_centreline(wells):
    mid  = (PLUME_ROWS[0]+PLUME_ROWS[1])//2
    x    = np.arange(GRID_NX)*CELL_SIZE
    s_tot= drawdown_field(wells)

    fig, ax = plt.subplots(figsize=(11,5))
    cmap = plt.get_cmap("tab10")
    for k, w in enumerate(wells):
        dm_k = -drawdown_field([w])[mid,:]
        ls   = "--" if w["type"]=="extraction" else ":"
        ax.plot(x, dm_k, lw=1.2, ls=ls, color=cmap(k), alpha=0.7,
                label=f"{w['type'][0].upper()} r={w['row']} c={w['col']} "
                      f"Q={w['Q']:.0f}")
    ax.plot(x, -s_tot[mid,:], lw=2.8, color="black",
            label="Total (superposition)")
    ax.axhline(MIN_DRAW_TARGET, color="red", ls="--", lw=1.5,
               label=f"Target {MIN_DRAW_TARGET} m")
    ax.axhline(0, color="gray", lw=0.7)
    ax.axvspan(PLUME_COLS[0]*CELL_SIZE, PLUME_COLS[1]*CELL_SIZE,
               alpha=0.12, color="gold", label="Plume")
    ax.set_xlabel("x [m]"); ax.set_ylabel("Drawdown magnitude –s [m]")
    ax.set_title(f"Drawdown profile — plume centreline (row {mid})")
    ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

plot_centreline(best_wells)


## 📊 Optimised Pumping Rates


In [ ]:
def plot_q_bars(wells):
    labels = [f"{'E' if w['type']=='extraction' else 'I'}{i+1}\n"
              f"({w['col']*CELL_SIZE:.0f},{w['row']*CELL_SIZE:.0f})"
              for i,w in enumerate(wells)]
    qs     = [w["Q"] for w in wells]
    colors = ["crimson" if w["type"]=="extraction" else "deepskyblue"
              for w in wells]

    fig, ax = plt.subplots(figsize=(9,4))
    bars = ax.bar(labels, qs, color=colors, edgecolor="black", lw=0.7)
    ax.axhline(0, color="black", lw=0.8)
    for lv, col, ls in [(Q_E_MAX,"crimson",":"),(Q_E_MIN,"crimson","--"),
                         (Q_I_MAX,"deepskyblue","--"),(Q_I_MIN,"deepskyblue",":")]:
        ax.axhline(lv, color=col, lw=1, ls=ls, alpha=0.5, label=f"{lv:.0f}")
    for bar,q in zip(bars,qs):
        ax.text(bar.get_x()+bar.get_width()/2, q+15*(1 if q>=0 else -1),
                f"{q:.0f}", ha="center", va="bottom" if q>=0 else "top",
                fontsize=9, fontweight="bold")
    ax.set_ylabel("Q [m³/day]")
    ax.set_title("Optimised pumping rates  (E=extraction, I=injection)\n"
                 "x-axis: (x m, y m) of well centre")
    ax.legend(fontsize=8, loc="upper right"); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout(); plt.show()

plot_q_bars(best_wells)
